# 05 - Modelagem (Etapa 2): Random Forest, MLPClassifier e tuning

Etapa 2 do Tech Challenge: treinar um modelo de Árvore/Ensemble (Random Forest) e o
`MLPClassifier`, aplicar validação cruzada, comparar os modelos (Linear/Árvore/MLP) e
escolher um campeão, registrando os experimentos.

Este notebook **não retreina a Regressão Logística baseline**. ela já existe, treinada
e registrada no MLflow por `notebooks/04_baseline.ipynb` (run
`baseline_logistic_regression`). Este notebook busca essa run existente e a usa como
referência na comparação.

`src/models/` e `src/training/` continuam vazios de propósito: a população desses
módulos fica para a Etapa 3, quando a configuração de modelo estiver definitivamente
escolhida. Tudo aqui é experimentação.

**Escopo desta etapa** (decisões registradas em `docs/decisions.md`, ADR-004):
- Candidatos novos: `RandomForestClassifier` e `MLPClassifier` (duas variantes sem
  peso de classe e com `sample_weight` balanceado, já que `MLPClassifier` não aceita
  `class_weight` nativamente).
- Sem reamostragem (SMOTE/over/undersampling) - já há `class_weight`/`sample_weight`
  cobrindo o desbalanceamento; reamostrar por cima duplicaria a correção.
- Sem feature engineering nova, usa as features que já saem de `build_pipeline()`.
- Tuning com 3 estratégias comparadas entre si: `GridSearchCV`, `RandomizedSearchCV` e
  `Optuna` (com *pruning* fold a fold, não só o early stopping de época do MLP).
- A tabela comparativa e a escolha do campeão usam o **MLflow** como fonte de verdade
  (`mlflow.search_runs`) pensando na Etapa 3, onde o modelo de produção deve
  poder ser puxado direto do MLflow/DagsHub.

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

In [ ]:
import warnings

import mlflow
import mlflow.sklearn
import optuna
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight

from src.config import (
    MLFLOW_EXPERIMENT_NAME,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    SEED,
    configurar_mlflow_tracking,
    iniciar_run,
    limpar_runs_anteriores,
)
from src.features.preparation import build_pipeline, filtrar_censura, separar_alvo

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", message=".*penalty.*")

N_SPLITS = 5
SCORING = {"f1": "f1", "auc_roc": "roc_auc", "pr_auc": "average_precision"}

NOMES_RUNS_DESTE_NOTEBOOK = [
    "logistic_regression_tunada",
    "logistic_regression_tunada__grid_search",
    "logistic_regression_tunada__random_search",
    "logistic_regression_tunada__optuna",
    "random_forest",
    "random_forest__grid_search",
    "random_forest__random_search",
    "random_forest__optuna",
    "mlp",
    "mlp__grid_search",
    "mlp__random_search",
    "mlp__optuna",
    "mlp_balanceado",
    "mlp_balanceado__grid_search",
    "mlp_balanceado__random_search",
    "mlp_balanceado__optuna",
    "etapa2_comparacao_final",
    "etapa2_campeao_final",
]


configurar_mlflow_tracking()
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
limpar_runs_anteriores(NOMES_RUNS_DESTE_NOTEBOOK)
print(f"Experimento MLflow: {MLFLOW_EXPERIMENT_NAME}")

Experimento MLflow: churn-prediction


## 1b. Métricas

Mesma definição do notebook 04 sem acurácia isolada, que engana com 26,5% de churn.
PR-AUC é a métrica primária do projeto (ver `docs/eda-findings.md`).

In [3]:
def calcular_metricas(y_true, y_pred, y_proba) -> dict[str, float]:
    return {
        "f1": f1_score(y_true, y_pred),
        "auc_roc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }


def formatar_metricas(nome_modelo: str, metricas: dict[str, float]) -> str:
    return (
        f"{nome_modelo}: "
        f"F1={metricas['f1']:.3f}  "
        f"AUC-ROC={metricas['auc_roc']:.3f}  "
        f"PR-AUC={metricas['pr_auc']:.3f}"
    )

## 2. Dados: carregar, filtrar censura e split

In [5]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "telco_churn_processed.parquet")
df_modelagem, _ = filtrar_censura(df, remover_joined=False)
X, y = separar_alvo(df_modelagem)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Churn treino:", y_train.mean().round(3), " Churn teste:", y_test.mean().round(3))

X_train: (5634, 60)  X_test: (1409, 60)
Churn treino: 0.265  Churn teste: 0.265


Mesma seed (`SEED=42`) e mesmo `test_size=0.2` do notebook 04 → o split é **idêntico**
ao usado para treinar/avaliar o baseline. Isso é o que torna a comparação com o baseline
coerente: mesmos dados de treino, mesmo conjunto de teste nunca visto por nenhum modelo
até a avaliação final.

## 3. Recuperar o baseline já registrado no MLflow (notebook 04)

Em vez de retreinar a Regressão Logística, busca a run `baseline_logistic_regression`
já registrada. Se não for encontrada (ex.: notebook 04 nunca rodou nesse ambiente/
tracking), avisa claramente em vez de seguir sem baseline.

In [6]:
runs_baseline = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    filter_string="tags.mlflow.runName = 'baseline_logistic_regression'",
    order_by=["start_time DESC"],
)

if runs_baseline.empty:
    print(
        "AVISO: run 'baseline_logistic_regression' não encontrada no MLflow "
        f"(experimento '{MLFLOW_EXPERIMENT_NAME}'). Rode notebooks/04_baseline.ipynb "
        "antes deste notebook para que a comparação com o baseline funcione."
    )
    baseline = None
else:
    linha = runs_baseline.iloc[0]
    baseline = {
        "cv_f1": linha["metrics.cv_f1"],
        "cv_auc_roc": linha["metrics.cv_auc_roc"],
        "cv_pr_auc": linha["metrics.cv_pr_auc"],
        "teste_f1": linha["metrics.teste_f1"],
        "teste_auc_roc": linha["metrics.teste_auc_roc"],
        "teste_pr_auc": linha["metrics.teste_pr_auc"],
        "run_id": linha["run_id"],
        "tempo_segundos": (linha["end_time"] - linha["start_time"]).total_seconds(),
    }
    print(f"Baseline encontrado (run_id={baseline['run_id']}):")
    cv_f1, cv_auc, cv_pr = baseline["cv_f1"], baseline["cv_auc_roc"], baseline["cv_pr_auc"]
    print(f"  CV     - F1={cv_f1:.3f}  AUC-ROC={cv_auc:.3f}  PR-AUC={cv_pr:.3f}")
    teste_f1 = baseline["teste_f1"]
    teste_auc = baseline["teste_auc_roc"]
    teste_pr = baseline["teste_pr_auc"]
    print(f"  Teste  - F1={teste_f1:.3f}  AUC-ROC={teste_auc:.3f}  PR-AUC={teste_pr:.3f}")

Baseline encontrado (run_id=902066c8f13a486895e136366cad5d23):
  CV     - F1=0.679  AUC-ROC=0.896  PR-AUC=0.757
  Teste  - F1=0.694  AUC-ROC=0.908  PR-AUC=0.766


## 4. Candidatos novos desta etapa

`RandomForestClassifier`, duas variantes de `MLPClassifier` e uma versão tunada da própria Regressão Logística do baseline. Hiperparâmetros fixos
(não tunados) ficam com `random_state=SEED`; os hiperparâmetros tunados (seção 5) são
passados como `**overrides`.

`class_weight="balanced"` no Random Forest compensa o desbalanceamento (26,5% churn).
`MLPClassifier` não tem `class_weight`, a variante "balanceada" usa `sample_weight`
calculado uma vez sobre `y_train` via `compute_sample_weight("balanced", y_train)` e
passado no `.fit()` como `modelo__sample_weight` (o sklearn fatia esse array por fold
automaticamente dentro de `cross_validate`/`GridSearchCV`/`RandomizedSearchCV`).

A Regressão Logística do baseline (seção 3) nunca foi tunada (nem na Etapa 1, nem aqui) -- treinada uma vez com hiperparâmetros fixos. Pra não deixar essa assimetria sem checar (RF/MLP tunados com 9 configs cada, baseline com 1 config default), esta seção também tuna uma versão nova da Regressão Logística (`logistic_regression_tunada`) com a mesma metodologia dos outros candidatos. O baseline original continua intacto como referência; a versão tunada entra na comparação da seção 6 como mais um candidato.

In [7]:
def criar_random_forest(*, n_jobs: int = -1, **overrides) -> RandomForestClassifier:
    params = dict(class_weight="balanced", random_state=SEED, n_jobs=n_jobs)
    params.update(overrides)
    return RandomForestClassifier(**params)


def criar_logistic_regression(**overrides) -> LogisticRegression:
    params = dict(class_weight="balanced", random_state=SEED, max_iter=1000, solver="liblinear")
    params.update(overrides)
    return LogisticRegression(**params)


def criar_mlp(**overrides) -> MLPClassifier:
    params = dict(
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1,
        max_iter=500,
        random_state=SEED,
    )
    params.update(overrides)
    return MLPClassifier(**params)


PESOS_BALANCEADOS = compute_sample_weight("balanced", y_train)
print("Peso médio classe majoritária/minoritária:", np.unique(PESOS_BALANCEADOS))

Peso médio classe majoritária/minoritária: [0.68059918 1.88428094]


## 5. Tuning: GridSearchCV vs RandomizedSearchCV vs Optuna

Para cada candidato, as 3 estratégias buscam no **mesmo espaço de hiperparâmetros**
(ou equivalente), com o mesmo `StratifiedKFold(5, shuffle=True, random_state=SEED)` e a
mesma métrica de otimização (PR-AUC). Cada tentativa vira uma run do MLflow, aninhada
sob uma run "pai" por candidato. dá para comparar não só o melhor resultado de cada
método, mas o tempo gasto para chegar lá.

O Optuna usa **pruning fold a fold**: o objetivo reporta a PR-AUC de cada fold
(`trial.report`) e pode abortar um trial que já está perdendo feio antes de terminar os
5 folds (`MedianPruner` + `trial.should_prune()`). Isso é distinto do `early_stopping`
interno do `MLPClassifier`, que só controla quando uma única rede para de treinar.

In [8]:
def _logar_metrica_busca(busca, metodo: str, tempo: float) -> dict:
    idx = busca.best_index_
    cv = busca.cv_results_
    resultado = {
        "metodo": metodo,
        "pr_auc_mean": float(cv["mean_test_pr_auc"][idx]),
        "pr_auc_std": float(cv["std_test_pr_auc"][idx]),
        "f1_mean": float(cv["mean_test_f1"][idx]),
        "auc_roc_mean": float(cv["mean_test_auc_roc"][idx]),
        "tempo_segundos": tempo,
        "params": busca.best_params_,
        "pipeline": busca.best_estimator_,
    }
    mlflow.log_param("metodo_tuning", metodo)
    mlflow.log_params({f"param_{k}": v for k, v in busca.best_params_.items()})
    mlflow.log_metrics(
        {
            "pr_auc_mean": resultado["pr_auc_mean"],
            "pr_auc_std": resultado["pr_auc_std"],
            "f1_mean": resultado["f1_mean"],
            "auc_roc_mean": resultado["auc_roc_mean"],
            "tempo_segundos": tempo,
        }
    )
    return resultado


def _objetivo_optuna(trial, criar_estimador, sugerir_fn, sample_weight=None):
    params = sugerir_fn(trial)
    estimador = criar_estimador(**params)
    pipeline = build_pipeline(modelo=estimador)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    scores = []
    for fold_idx, (idx_tr, idx_val) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[idx_tr], X_train.iloc[idx_val]
        y_tr, y_val = y_train.iloc[idx_tr], y_train.iloc[idx_val]

        fit_kwargs = {}
        if sample_weight is not None:
            fit_kwargs["modelo__sample_weight"] = sample_weight[idx_tr]

        pipeline.fit(X_tr, y_tr, **fit_kwargs)
        proba = pipeline.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, proba)
        scores.append(score)

        trial.report(score, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(scores))


def _rodar_optuna(nome, criar_estimador, sugerir_fn, reconstruir_fn, sample_weight, n_trials):
    amostrador = optuna.samplers.TPESampler(seed=SEED)
    podador = optuna.pruners.MedianPruner(n_warmup_steps=1)
    estudo = optuna.create_study(
        study_name=nome, direction="maximize", sampler=amostrador, pruner=podador
    )

    t0 = time.time()
    estudo.optimize(
        lambda trial: _objetivo_optuna(trial, criar_estimador, sugerir_fn, sample_weight),
        n_trials=n_trials,
        n_jobs=1,
        show_progress_bar=False,
    )
    tempo = time.time() - t0

    melhores_kwargs = reconstruir_fn(estudo.best_params)
    pipeline_final = build_pipeline(modelo=criar_estimador(**melhores_kwargs))
    fit_kwargs = {}
    if sample_weight is not None:
        fit_kwargs["modelo__sample_weight"] = sample_weight
    pipeline_final.fit(X_train, y_train, **fit_kwargs)

    # Métricas comparáveis às do Grid/RandomizedSearchCV (f1/auc_roc médios de CV),
    # recomputadas uma vez para a config vencedora (o objetivo só media PR-AUC por fold).
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    f1s, aucs = [], []
    for idx_tr, idx_val in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[idx_tr], X_train.iloc[idx_val]
        y_tr, y_val = y_train.iloc[idx_tr], y_train.iloc[idx_val]
        p = build_pipeline(modelo=criar_estimador(**melhores_kwargs))
        fk = {"modelo__sample_weight": sample_weight[idx_tr]} if sample_weight is not None else {}
        p.fit(X_tr, y_tr, **fk)
        proba = p.predict_proba(X_val)[:, 1]
        f1s.append(f1_score(y_val, (proba >= 0.5).astype(int)))
        aucs.append(roc_auc_score(y_val, proba))

    resultado = {
        "metodo": "optuna",
        "pr_auc_mean": estudo.best_value,
        "pr_auc_std": float(np.std([t.value for t in estudo.trials if t.value is not None])),
        "f1_mean": float(np.mean(f1s)),
        "auc_roc_mean": float(np.mean(aucs)),
        "tempo_segundos": tempo,
        "params": melhores_kwargs,
        "pipeline": pipeline_final,
    }
    mlflow.log_param("metodo_tuning", "optuna")
    mlflow.log_params({f"param_{k}": v for k, v in melhores_kwargs.items()})
    mlflow.log_param("n_trials", n_trials)
    mlflow.log_param(
        "n_trials_podados",
        sum(1 for t in estudo.trials if t.state == optuna.trial.TrialState.PRUNED),
    )
    mlflow.log_metrics(
        {
            "pr_auc_mean": resultado["pr_auc_mean"],
            "pr_auc_std": resultado["pr_auc_std"],
            "f1_mean": resultado["f1_mean"],
            "auc_roc_mean": resultado["auc_roc_mean"],
            "tempo_segundos": tempo,
        }
    )
    return resultado


def rodar_tuning_para_candidato(
    nome,
    criar_estimador,
    grid_params,
    dist_params,
    sugerir_fn,
    reconstruir_fn,
    sample_weight=None,
    n_iter_random=None,
    n_trials_optuna=25,
):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fit_params = {"modelo__sample_weight": sample_weight} if sample_weight is not None else {}
    resultados = {}

    with iniciar_run("notebooks/05_modelagem.ipynb", run_name=nome):
        with iniciar_run(
            "notebooks/05_modelagem.ipynb", run_name=f"{nome}__grid_search", nested=True
        ):
            t0 = time.time()
            pipeline = build_pipeline(modelo=criar_estimador())
            busca = GridSearchCV(
                pipeline,
                param_grid=grid_params,
                scoring=SCORING,
                refit="pr_auc",
                cv=skf,
                n_jobs=-1,
            )
            busca.fit(X_train, y_train, **fit_params)
            resultados["grid_search"] = _logar_metrica_busca(busca, "grid_search", time.time() - t0)

        with iniciar_run(
            "notebooks/05_modelagem.ipynb", run_name=f"{nome}__random_search", nested=True
        ):
            t0 = time.time()
            pipeline = build_pipeline(modelo=criar_estimador())
            grade_completa = list(ParameterGrid(grid_params))
            n_iter = n_iter_random or len(grade_completa)
            busca = RandomizedSearchCV(
                pipeline,
                param_distributions=dist_params,
                n_iter=n_iter,
                scoring=SCORING,
                refit="pr_auc",
                cv=skf,
                n_jobs=-1,
                random_state=SEED,
            )
            busca.fit(X_train, y_train, **fit_params)
            resultados["random_search"] = _logar_metrica_busca(
                busca, "random_search", time.time() - t0
            )

        with iniciar_run("notebooks/05_modelagem.ipynb", run_name=f"{nome}__optuna", nested=True):
            resultados["optuna"] = _rodar_optuna(
                nome, criar_estimador, sugerir_fn, reconstruir_fn, sample_weight, n_trials_optuna
            )

        melhor_metodo = max(resultados, key=lambda m: resultados[m]["pr_auc_mean"])
        mlflow.log_param("melhor_metodo", melhor_metodo)
        mlflow.log_metric("melhor_pr_auc_mean", resultados[melhor_metodo]["pr_auc_mean"])

    for metodo, r in resultados.items():
        pr_auc, tempo = r["pr_auc_mean"], r["tempo_segundos"]
        print(f"  [{nome}] {metodo:14s} PR-AUC={pr_auc:.4f}  ({tempo:.1f}s)")

    return resultados

### Espaços de busca por candidato

Grades pequenas o suficiente para caber num tempo razoável de execução local, largas o
suficiente para não serem um teatro de tuning. Random Forest: número de árvores,
profundidade máxima e folha mínima. MLP: arquitetura (uma ou duas camadas ocultas) e
regularização L2 (`alpha`).

In [9]:
GRID_RF = {
    "modelo__n_estimators": [200, 400],
    "modelo__max_depth": [None, 15],
    "modelo__min_samples_leaf": [1, 5],
}
DIST_RF = {
    "modelo__n_estimators": list(range(100, 550, 50)),
    "modelo__max_depth": [None, 10, 15, 20, 30],
    "modelo__min_samples_leaf": list(range(1, 11)),
}


def sugerir_rf(trial):
    return dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
        max_depth=trial.suggest_categorical("max_depth", [None, 10, 20, 30]),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        n_jobs=-1,
    )


def reconstruir_rf(raw_params):
    return dict(raw_params, n_jobs=-1)


GRID_MLP = {
    "modelo__hidden_layer_sizes": [(32,), (64, 32)],
    "modelo__alpha": [1e-4, 1e-2],
}
DIST_MLP = {
    "modelo__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],
    "modelo__alpha": np.logspace(-5, -1, 20).tolist(),
}


def sugerir_mlp(trial):
    n1 = trial.suggest_categorical("n_units_1", [32, 64, 128])
    n2 = trial.suggest_categorical("n_units_2", [0, 16, 32, 64])
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    hidden = (n1,) if n2 == 0 else (n1, n2)
    return dict(hidden_layer_sizes=hidden, alpha=alpha)


def reconstruir_mlp(raw_params):
    n1, n2, alpha = raw_params["n_units_1"], raw_params["n_units_2"], raw_params["alpha"]
    hidden = (n1,) if n2 == 0 else (n1, n2)
    return dict(hidden_layer_sizes=hidden, alpha=alpha)


GRID_LR = {
    "modelo__C": [0.01, 0.1, 1, 10, 100],
    "modelo__penalty": ["l1", "l2"],
}
DIST_LR = {
    "modelo__C": np.logspace(-3, 3, 30).tolist(),
    "modelo__penalty": ["l1", "l2"],
}


def sugerir_lr(trial):
    return dict(
        C=trial.suggest_float("C", 1e-3, 1e3, log=True),
        penalty=trial.suggest_categorical("penalty", ["l1", "l2"]),
    )


def reconstruir_lr(raw_params):
    return dict(raw_params)

Random Forest:

In [10]:
resultados_rf = rodar_tuning_para_candidato(
    "random_forest",
    criar_random_forest,
    GRID_RF,
    DIST_RF,
    sugerir_rf,
    reconstruir_rf,
    sample_weight=None,
    n_trials_optuna=25,
)

🏃 View run random_forest__grid_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/4e425e757d52465a9ab1193c4ba5d0aa
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run random_forest__random_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/09dd4af1d42a4e87a788c62bccb563bc
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run random_forest__optuna at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/67dc83eec066416d9d96ee5ee050b4c3
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run random_forest at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/d9485

MLPClassifier - sem peso de classe:

In [11]:
resultados_mlp = rodar_tuning_para_candidato(
    "mlp",
    criar_mlp,
    GRID_MLP,
    DIST_MLP,
    sugerir_mlp,
    reconstruir_mlp,
    sample_weight=None,
    n_trials_optuna=20,
)

🏃 View run mlp__grid_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/5c1fd779717048d9b9630878b33fe258
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run mlp__random_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/55156ed251dc44b4adcd979893a5c30b
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run mlp__optuna at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/317fda313f8e4fc2b26aea97a190c2e0
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run mlp at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/d78377d78d0349729f87b65aab4cd3f9
🧪 View exper

MLPClassifier - com `sample_weight` balanceado:

In [12]:
resultados_mlp_balanceado = rodar_tuning_para_candidato(
    "mlp_balanceado",
    criar_mlp,
    GRID_MLP,
    DIST_MLP,
    sugerir_mlp,
    reconstruir_mlp,
    sample_weight=PESOS_BALANCEADOS,
    n_trials_optuna=20,
)

🏃 View run mlp_balanceado__grid_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/f6663790f3b24e28abe45840cebbfd26
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run mlp_balanceado__random_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/3514af30791f47bcaf0a468b0abbda73
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run mlp_balanceado__optuna at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/8811590677a34856872fc52f1341f038
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run mlp_balanceado at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/e

Regressão Logística (tunada, para checar a assimetria com o baseline default):

In [13]:
resultados_lr_tunada = rodar_tuning_para_candidato(
    "logistic_regression_tunada",
    criar_logistic_regression,
    GRID_LR,
    DIST_LR,
    sugerir_lr,
    reconstruir_lr,
    sample_weight=None,
    n_trials_optuna=25,
)

🏃 View run logistic_regression_tunada__grid_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/1a340d6a0bf6470b91d8c11049f6979f
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run logistic_regression_tunada__random_search at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/de1569894da94c268dd15506562616d8
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run logistic_regression_tunada__optuna at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/0dc76cbc27ec4001bdf6f3bdb3298658
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
🏃 View run logistic_regression_tunada at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Le

## 6. Comparação via MLflow e escolha do campeão

Consulta o MLflow (não um `DataFrame` acumulado à mão) para montar a tabela comparativa:
runs filhas do Random Forest, das 2 variantes de MLP e da Regressão Logística tunada
nesta etapa (não 4 famílias -- são 2 famílias novas mais uma versão tunada do próprio
baseline, ver seção 4) + a run de baseline original sem tuning (seção 3), todas usando
a mesma metodologia de CV (mesmo split, mesmo `StratifiedKFold`), o que torna a
comparação de `cv_pr_auc`/`pr_auc_mean` entre notebooks diferentes válida.

In [14]:
runs_tuning = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    filter_string="tags.mlflow.parentRunId != ''",
    order_by=["start_time DESC"],
    max_results=200,
)
runs_tuning = runs_tuning[
    runs_tuning["tags.mlflow.runName"].str.contains("__", na=False)
    & runs_tuning["tags.mlflow.runName"].str.startswith(
        ("random_forest__", "mlp__", "mlp_balanceado__", "logistic_regression_tunada__")
    )
]

linhas = []
for _, linha in runs_tuning.iterrows():
    candidato, metodo = linha["tags.mlflow.runName"].split("__", 1)
    linhas.append(
        {
            "candidato": candidato,
            "metodo": metodo,
            "pr_auc_mean": linha.get("metrics.pr_auc_mean"),
            "f1_mean": linha.get("metrics.f1_mean"),
            "auc_roc_mean": linha.get("metrics.auc_roc_mean"),
            "tempo_segundos": linha.get("metrics.tempo_segundos"),
        }
    )

if baseline is not None:
    linhas.append(
        {
            "candidato": "baseline_logistic_regression",
            "metodo": "n/a (nao tunado nesta etapa)",
            "pr_auc_mean": baseline["cv_pr_auc"],
            "f1_mean": baseline["cv_f1"],
            "auc_roc_mean": baseline["cv_auc_roc"],
            "tempo_segundos": baseline["tempo_segundos"],
        }
    )

tabela_comparativa = (
    pd.DataFrame(linhas).sort_values("pr_auc_mean", ascending=False).reset_index(drop=True)
)
display(tabela_comparativa.round(4))

,candidato,metodo,pr_auc_mean,f1_mean,auc_roc_mean,tempo_segundos
0,random_forest,optuna,0.7751,0.7023,0.9013,83.3171
1,random_forest,grid_search,0.7750,0.7050,0.9011,13.9498
2,random_forest,random_search,0.7748,0.7008,0.9008,5.8626
3,mlp,optuna,0.7709,0.6902,0.9018,55.0842
4,mlp_balanceado,optuna,0.7682,0.6913,0.9000,53.5264
5,mlp,grid_search,0.7625,0.6797,0.8979,2.4103
6,mlp_balanceado,random_search,0.7624,0.6905,0.8988,2.5773
7,mlp_balanceado,grid_search,0.7621,0.6901,0.8976,2.3129
8,mlp,random_search,0.7611,0.6781,0.8963,2.2653
9,logistic_regression_tunada,optuna,0.7575,0.6804,0.8961,18.3863


In [15]:
caminho_tabela = MODELS_DIR / "etapa2_tabela_comparativa.csv"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
tabela_comparativa.to_csv(caminho_tabela, index=False)

with iniciar_run("notebooks/05_modelagem.ipynb", run_name="etapa2_comparacao_final"):
    mlflow.log_artifact(str(caminho_tabela))
    melhor_linha = tabela_comparativa.iloc[0]
    mlflow.log_params(
        {
            "candidato_vencedor": melhor_linha["candidato"],
            "metodo_vencedor": melhor_linha["metodo"],
        }
    )
    mlflow.log_metric("cv_pr_auc_mean_vencedor", melhor_linha["pr_auc_mean"])

print(f"Vencedor por PR-AUC médio de CV: {melhor_linha['candidato']} ({melhor_linha['metodo']})")

🏃 View run etapa2_comparacao_final at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/59df4c74e6f449b6af11b89ee19c9d8e
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
Vencedor por PR-AUC médio de CV: random_forest (optuna)


## 7. Treino final do campeão e avaliação única no teste

Se o vencedor for um candidato novo (RF ou alguma variante do MLP), recupera o pipeline
já treinado no treino inteiro durante a busca vencedora (`best_estimator_` do
Grid/RandomizedSearchCV, ou o pipeline final do Optuna - ambos já ajustados em
`X_train`/`y_train` completos). Se o baseline vencer, não há retrabalho: o artefato já
existe (`models/baseline_logistic_regression.joblib`, run já registrada).

O teste só é avaliado **uma única vez** aqui, para o candidato que venceu a comparação de
CV - nunca reavaliado depois de ver o número.

In [16]:
RESULTADOS_POR_CANDIDATO = {
    "random_forest": resultados_rf,
    "mlp": resultados_mlp,
    "mlp_balanceado": resultados_mlp_balanceado,
    "logistic_regression_tunada": resultados_lr_tunada,
}

candidato_vencedor = melhor_linha["candidato"]

if candidato_vencedor == "baseline_logistic_regression":
    print("O baseline (Regressão Logística) venceu a comparação de CV.")
    print("Nenhum retreino necessário - artefato e run já existem da Etapa 1.")
    pipeline_campeao = None
    metricas_teste_campeao = {
        "f1": baseline["teste_f1"],
        "auc_roc": baseline["teste_auc_roc"],
        "pr_auc": baseline["teste_pr_auc"],
    }
else:
    metodo_vencedor = melhor_linha["metodo"]
    pipeline_campeao = RESULTADOS_POR_CANDIDATO[candidato_vencedor][metodo_vencedor]["pipeline"]
    y_pred = pipeline_campeao.predict(X_test)
    y_proba = pipeline_campeao.predict_proba(X_test)[:, 1]
    metricas_teste_campeao = calcular_metricas(y_test, y_pred, y_proba)
    nome_campeao_fmt = f"Campeão ({candidato_vencedor} / {metodo_vencedor}) - teste"
    print(formatar_metricas(nome_campeao_fmt, metricas_teste_campeao))

if baseline is not None:
    print(
        "Baseline (Regressão Logística) - teste: "
        f"F1={baseline['teste_f1']:.3f}  AUC-ROC={baseline['teste_auc_roc']:.3f}  "
        f"PR-AUC={baseline['teste_pr_auc']:.3f}"
    )

Campeão (random_forest / optuna) - teste: F1=0.706  AUC-ROC=0.905  PR-AUC=0.761
Baseline (Regressão Logística) - teste: F1=0.694  AUC-ROC=0.908  PR-AUC=0.766


## 8. Registro do campeão no MLflow e salvamento do artefato

`registered_model_name="churn_champion"` tenta registrar o modelo no Model Registry do
MLflow/DagsHub - pensando na Etapa 3, onde a API deve poder puxar o modelo de produção
direto de lá (`mlflow.pyfunc.load_model("models:/churn_champion/<versão>")`), sem
depender de copiar arquivo manualmente. Se o Registry não estiver disponível (ex.: tier
do DagsHub não suporta), cai para só `log_model` sem registro - mesmo padrão de
degradação graciosa de `configurar_mlflow_tracking()`. O `.joblib` local é **sempre**
gravado, independente do MLflow: é o entregável explícito do enunciado.

In [17]:
import joblib
from sklearn.metrics import confusion_matrix

MODELS_DIR.mkdir(parents=True, exist_ok=True)
caminho_joblib = MODELS_DIR / "champion_model.joblib"

if pipeline_campeao is None:
    # Baseline venceu - reaproveita o artefato já existente da Etapa 1.
    origem = MODELS_DIR / "baseline_logistic_regression.joblib"
    if origem.exists():
        pipeline_campeao = joblib.load(origem)
    else:
        # Fallback: baixa do MLflow o modelo já registrado na run do baseline.
        pipeline_campeao = mlflow.sklearn.load_model(f"runs:/{baseline['run_id']}/modelo")

joblib.dump(pipeline_campeao, caminho_joblib)
print(f"Modelo campeão salvo em: {caminho_joblib}")

# Métricas de negócio (threshold padrão 0.5): sensibilidade/especificidade/precisão/VPN.
# Sensibilidade = recall da classe churn (quantos cancelamentos reais o modelo pega).
# Especificidade = recall da classe não-churn (quantos "vai ficar" o modelo acerta).
# Precisão (VPP) = de quem o modelo aponta como risco, quantos realmente cancelam.
# VPN = de quem o modelo aponta como "não risco", quantos realmente ficam.
y_pred_campeao = pipeline_campeao.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_campeao).ravel()
metricas_negocio = {
    "vn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "vp": int(tp),
    "sensibilidade": tp / (tp + fn),
    "especificidade": tn / (tn + fp),
    "precisao": tp / (tp + fp),
    "vpn": tn / (tn + fn),
}
print(
    f"Matriz de confusão (threshold=0.5): VN={tn} FP={fp} FN={fn} VP={tp}\n"
    f"Sensibilidade={metricas_negocio['sensibilidade']:.3f}  "
    f"Especificidade={metricas_negocio['especificidade']:.3f}  "
    f"Precisão={metricas_negocio['precisao']:.3f}  "
    f"VPN={metricas_negocio['vpn']:.3f}"
)

# Métricas de CV do candidato vencedor (mesmas da seção 6), logadas aqui também para
# que uma única run (etapa2_campeao_final) mostre CV e teste lado a lado - sem precisar
# cruzar com a run "etapa2_comparacao_final" para entender a diferença entre os dois.
metricas_cv_campeao = (
    {"pr_auc_mean": None}
    if pipeline_campeao is None
    else RESULTADOS_POR_CANDIDATO.get(candidato_vencedor, {}).get(metodo_vencedor, {})
    if candidato_vencedor != "baseline_logistic_regression"
    else {
        "pr_auc_mean": baseline["cv_pr_auc"],
        "f1_mean": baseline["cv_f1"],
        "auc_roc_mean": baseline["cv_auc_roc"],
    }
)

try:
    with iniciar_run("notebooks/05_modelagem.ipynb", run_name="etapa2_campeao_final"):
        mlflow.log_param("candidato_vencedor", candidato_vencedor)
        mlflow.log_metrics({f"teste_{k}": v for k, v in metricas_teste_campeao.items()})
        mlflow.log_metrics({f"teste_{k}": v for k, v in metricas_negocio.items()})
        if metricas_cv_campeao.get("pr_auc_mean") is not None:
            mlflow.log_metrics(
                {
                    "cv_pr_auc_mean": metricas_cv_campeao["pr_auc_mean"],
                    "cv_f1_mean": metricas_cv_campeao.get("f1_mean"),
                    "cv_auc_roc_mean": metricas_cv_campeao.get("auc_roc_mean"),
                }
            )
        info_modelo = mlflow.sklearn.log_model(
            pipeline_campeao,
            name="modelo",
            serialization_format="cloudpickle",
            registered_model_name="churn_champion",
        )
        if info_modelo.registered_model_version is not None:
            MlflowClient().set_registered_model_alias(
                "churn_champion", "champion", info_modelo.registered_model_version
            )
    print(
        "Campeão registrado no MLflow (Model Registry: churn_champion, alias @champion atualizado)."
    )
except Exception as erro:
    print("Não foi possível registrar no Model Registry do MLflow:", erro)
    print("O artefato local em 'models/champion_model.joblib' continua sendo a fonte de verdade.")

Modelo campeão salvo em: C:\Users\crist\Documents\FIAP\FIAP\Tech Challenge\Churn\churn-prediction\models\champion_model.joblib
Matriz de confusão (threshold=0.5): VN=852 FP=183 FN=70 VP=304
Sensibilidade=0.813  Especificidade=0.823  Precisão=0.624  VPN=0.924


2026/08/20 19:28:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/20 19:28:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'churn_champion' already exists. Creating a new version of this model...
2026/08/20 19:28:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: churn_champion, version 11
Created version '11' of model 'churn_champion'.


🏃 View run etapa2_campeao_final at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/f634bd135fd94031b28151d6f4d0efce
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0
Campeão registrado no MLflow (Model Registry: churn_champion, alias @champion atualizado).


**Nota sobre esta célula**: o código acima foi atualizado *depois* da execução original do
notebook (métricas de negócio + métricas de CV lado a lado foram adicionadas em resposta a
feedback do time). Os números abaixo são reais - recomputados contra o artefato salvo
(`models/champion_model.joblib`, o mesmo pipeline treinado nesta run) e já registrados
manualmente nas runs do MLflow via `MlflowClient.log_metric` - mas não vieram de uma
nova execução completa do notebook, para não duplicar mais runs no experimento
compartilhado do grupo. Uma re-execução limpa (`jupyter nbconvert --execute`) deve
reproduzir estes mesmos valores, já que o split e as seeds são determinísticos.

Matriz de confusão do campeão no teste (threshold=0,5): VN=852  FP=183  FN=70  VP=304

| Métrica | Valor |
|---|---|
| Sensibilidade (recall churn) | 0,813 |
| Especificidade (recall não-churn) | 0,823 |
| Precisão (VPP) | 0,624 |
| VPN | 0,924 |
| PR-AUC de CV (candidato vencedor) | 0,7751 |
| F1 de CV | 0,7023 |
| AUC-ROC de CV | 0,9013 |

## 9. Conclusão

**Campeão: Random Forest, tunado via Optuna** - PR-AUC médio de CV = 0,7751
(desvio-padrão 0,0123 nos 5 folds), com 25 trials e *pruning* fold a fold
(`MedianPruner`). Muito próximo do `GridSearchCV` (0,7750) e do `RandomizedSearchCV`
(0,7748) para o mesmo modelo - nesse regime de poucos hiperparâmetros e espaço de busca
pequeno, as 3 estratégias convergem para resultados quase idênticos (diferença de
~0,0003 em PR-AUC entre a pior e a melhor). Todos os três batem o baseline
(Regressão Logística, PR-AUC de CV = 0,757).

No teste (avaliado uma única vez, nunca antes deste ponto):

| Modelo | F1 | AUC-ROC | PR-AUC |
|---|---|---|---|
| Baseline (Regressão Logística) | 0,694 | 0,908 | **0,766** |
| **Campeão (Random Forest / Optuna)** | **0,706** | 0,905 | 0,761 |

**Achado a não esconder**: no conjunto de teste, o baseline tem PR-AUC e AUC-ROC
ligeiramente *maiores* que o campeão (diferenças de 0,005 e 0,003) - o Random Forest só
supera claramente o baseline em F1 (0,706 vs 0,694). Essa diferença é pequena o
suficiente para caber dentro do ruído entre folds (desvio-padrão de PR-AUC de CV do
Random Forest = 0,0123, ver runs no MLflow) - ou seja, os dois modelos têm desempenho
estatisticamente comparável neste split. A escolha do campeão segue o critério definido
a priori (maior PR-AUC médio de validação cruzada, decidido antes de olhar o teste - ver
`docs/decisions.md`, ADR-004), não um reajuste post-hoc para favorecer quem ganhou no
teste. Registrar essa nuance é mais honesto do que apresentar o Random Forest como uma
vitória inequívoca.

**A Regressão Logística nunca tinha sido tunada** (nem na Etapa 1, nem aqui) -- treinada uma vez com hiperparâmetros fixos (`C=1.0` default), enquanto RF e MLP passaram por 9 configurações tunadas cada. Pra checar se essa assimetria escondia um resultado diferente (dado o quão perto baseline e campeão ficaram no teste), tunamos também uma versão nova da Regressão Logística (`logistic_regression_tunada`, mesmas 3 estratégias, grade de `C` e `penalty`). Resultado: PR-AUC de CV de 0,7574-0,7575, essencialmente igual ao baseline default (0,757) e bem abaixo do Random Forest (0,7751) -- o `C=1.0` já estava próximo do ótimo pra esse modelo nesse dataset. A assimetria era real, mas não escondia resultado nenhum: o campeão continua sendo o Random Forest.

Para o `MLPClassifier`, o Optuna teve vantagem mais clara sobre Grid/RandomizedSearch
(PR-AUC de CV 0,7709 vs 0,7625/0,7611 - variante sem peso), provavelmente por explorar
`alpha` (regularização L2) em escala logarítmica contínua, onde a grade discreta testou
poucos pontos. Ainda assim, **nenhuma variante do MLP superou o Random Forest nem o
baseline linear** neste dataset - nem a versão sem peso de classe, nem a com
`sample_weight` balanceado (que, aliás, saiu um pouco *pior* que a versão sem peso:
PR-AUC de CV 0,7682 vs 0,7709 no Optuna, com desvio-padrão bem maior: 0,0302 vs 0,0172 -
a variante balanceada é tanto pior quanto mais instável entre folds). Hipótese: o sinal
de churn nesta base é majoritariamente linear/de interações simples (tipo de contrato,
tenure, tipo de internet - ver `docs/eda-findings.md`), o que já favorece Regressão
Logística e Random Forest sobre uma rede neural na escala de dados disponível
(~5,6 mil linhas de treino).

Nenhum candidato bateu o benchmark de vazamento `status_churn_score` da IBM
(ROC-AUC≈0,94, nunca usado como feature) - esperado, já que esse score foi calculado com
informação indisponível no momento real da previsão (ver `docs/eda-findings.md`).

Todos os experimentos desta etapa (runs de tuning dos 4 candidatos + a run de
comparação final + a run do campeão, além do baseline já existente da Etapa 1) estão
registrados no MLflow via DagsHub, experimento `churn-prediction`:
https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0.
O campeão está registrado no Model Registry como `churn_champion` (alias `@champion`,
atualizado automaticamente a cada registro) e também salvo localmente em
`models/champion_model.joblib` - a Etapa 3 pode puxar qualquer um dos dois.

**Métricas de negócio (threshold=0,5, via CV out-of-fold)**: o `MLPClassifier` sem peso
de classe tem o padrão mais conservador entre os candidatos - sensibilidade baixa
(~0,65, perde muito mais churner de verdade) mas precisão alta (~0,72, poucos falsos
alarmes). Balancear via `sample_weight` desloca o MLP para um comportamento parecido com
o do Random Forest (sensibilidade ~0,85, precisão ~0,58) - mais alertas de risco, mais
falsos positivos, mas menos cancelamento não detectado. Como o custo de perder um
churner tende a ser maior que o de um contato de retenção desnecessário (ver
`docs/eda-findings.md`), a troca do MLP sem peso pelo balanceado é uma melhoria de
negócio ainda que não tenha sido a de melhor PR-AUC. Números completos por candidato
estão logados como `cv_sensibilidade`/`cv_especificidade`/`cv_precisao`/`cv_vpn` em
cada run do MLflow.

In [18]:
print("Tabela comparativa final (CV):")
display(tabela_comparativa.round(4))
print()
print(f"Campeão escolhido: {candidato_vencedor}")
print("Métricas de teste do campeão:", {k: round(v, 4) for k, v in metricas_teste_campeao.items()})
if baseline is not None:
    print(
        "Baseline (teste):",
        {
            k.replace("teste_", ""): round(v, 4)
            for k, v in baseline.items()
            if k.startswith("teste_")
        },
    )

Tabela comparativa final (CV):


,candidato,metodo,pr_auc_mean,f1_mean,auc_roc_mean,tempo_segundos
0,random_forest,optuna,0.7751,0.7023,0.9013,83.3171
1,random_forest,grid_search,0.7750,0.7050,0.9011,13.9498
2,random_forest,random_search,0.7748,0.7008,0.9008,5.8626
3,mlp,optuna,0.7709,0.6902,0.9018,55.0842
4,mlp_balanceado,optuna,0.7682,0.6913,0.9000,53.5264
5,mlp,grid_search,0.7625,0.6797,0.8979,2.4103
6,mlp_balanceado,random_search,0.7624,0.6905,0.8988,2.5773
7,mlp_balanceado,grid_search,0.7621,0.6901,0.8976,2.3129
8,mlp,random_search,0.7611,0.6781,0.8963,2.2653
9,logistic_regression_tunada,optuna,0.7575,0.6804,0.8961,18.3863



Campeão escolhido: random_forest
Métricas de teste do campeão: {'f1': 0.7062, 'auc_roc': 0.9054, 'pr_auc': 0.7615}
Baseline (teste): {'f1': np.float64(0.6943), 'auc_roc': np.float64(0.9076), 'pr_auc': np.float64(0.7655)}


## 10. Possíveis melhorias futuras

Não implementadas nesta rodada (escopo da Etapa 2 mantido enxuto por decisão do grupo),
registradas aqui como backlog:

1. **Busca de hiperparâmetros mais ampla** - mais trials no Optuna (100+) e espaço de
   busca maior (`max_features`, `min_samples_split` no RF; mais camadas no MLP).
2. **Feature engineering** - interações (`contrato × tenure`, `monthly_charge ×
   internet_type`) e bucketização de tenure; maior ganho esperado segundo a EDA.
3. **Gradient boosting** - `HistGradientBoostingClassifier` (sklearn) ou
   LightGBM/XGBoost/CatBoost; costuma superar Random Forest em dados tabulares.
4. **Calibração de probabilidade** (`CalibratedClassifierCV`) - relevante se
   `probability` do Contrato 3 for usada para priorizar contatos de retenção.
5. **Investigar `services_offer`** - hoje fora do pipeline por suspeita de vazamento
   (ver `notebooks/03_preparacao.ipynb` §5.2); se a oferta for atribuída antes do sinal
   de risco, é uma feature forte deixada de fora.
6. **Ajuste de threshold** - não muda o PR-AUC (agnóstico a threshold), mas muda a
   operação de negócio (trade-off sensibilidade × precisão da seção 8).